<a href="https://colab.research.google.com/github/Ektyagi/employee-salary-prediction/blob/main/EmployeeSalaryPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('/content/Salary Data.csv')

In [19]:
df.head()

,Age,Gender,Education Level,Job Title,Years of Experience,Salary
0,32.0,Male,Bachelor's,Software Engineer,5.0,90000.0
1,28.0,Female,Master's,Data Analyst,3.0,65000.0
2,45.0,Male,PhD,Senior Manager,15.0,150000.0
3,36.0,Female,Bachelor's,Sales Associate,7.0,60000.0
4,52.0,Male,Master's,Director,20.0,200000.0


In [16]:
df.shape

(324, 6)

In [12]:
df.isna().sum()

,0
Age,0
Gender,0
Education Level,0
Job Title,0
Years of Experience,0
Salary,0


In [11]:
df.dropna(inplace=True)

In [15]:
df.drop_duplicates(inplace=True)

In [5]:
from sklearn.model_selection import train_test_split

In [17]:
y=df['Salary']
X=df.drop('Salary',axis=1)

In [18]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [22]:
X['Job Title'].nunique()

174

In [47]:
# Function for comparing mean-absolute-error for models
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

def score_dataset(X_train,X_test,y_train,y_test):
  model1 = LinearRegression()
  model1.fit(X_train,y_train)
  pred = model1.predict(X_test)

  model2 = XGBRegressor()
  model2.fit(X_train,y_train)
  pred2 = model2.predict(X_test)
  print('MAE from XGB =',mean_absolute_error(pred2,y_test))
  print('MAE from LinearRegressor =',mean_absolute_error(pred,y_test))
  return None

## Approach 1: Dropping categorical data

In [28]:
X_train_drop = X_train.select_dtypes(exclude=['object'])
X_test_drop = X_test.select_dtypes(exclude=['object'])


In [48]:
print('MAE from Dropping all Categorical Data columns:')
print(score_dataset(X_train_drop,X_test_drop,y_train,y_test))

MAE from Dropping all Categorical Data columns:
MAE from XGB = 14692.388040865384
MAE from LinearRegressor = 12725.16585212421
None


In [38]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder

In [ ]:
#Label Encoder

le = LabelEncoder()
X_train_le = X_train.drop('Job Title',axis=1)
X_test_le = X_test.drop('Job Title',axis=1)
X_train_le['Gender'] = le.fit_transform(X_train_le['Gender'])
X_test_le['Gender'] = le.transform(X_test_le['Gender'])


In [ ]:
# Ordinal Encoder

oe = OrdinalEncoder()
X_train_le['Education Level'] = oe.fit_transform(X_train_le[['Education Level']])
X_test_le['Education Level'] = oe.transform(X_test_le[['Education Level']])


In [49]:
# Checking MAE after deleting Job title column and including other two categorical columns after labelling them

print('MAE after excluding JOB TITLE and including other two categorical columns:')
print(score_dataset(X_train_le,X_test_le,y_train,y_test))

MAE after excluding JOB TITLE and including other two categorical columns:
MAE from XGB = 10272.689663461539
MAE from LinearRegressor = 10216.184049021076
None


In [ ]:
# Directly applying one hot encoding to the Job Title column

oh = OneHotEncoder(handle_unknown='ignore',sparse_output=False)
X_train_col = pd.DataFrame(oh.fit_transform(X_train[['Job Title']]))
X_test_col = pd.DataFrame(oh.transform(X_test[['Job Title']]))

#One hot encoding removed index, so adding them
X_train_col.index = X_train.index
X_test_col.index = X_test.index

#Now adding these one hot encoded columns with other numerical columns, they are already present in X_train_le

X_train_oh = pd.concat([X_train_col,X_train_le],axis=1)
X_test_oh = pd.concat([X_test_col,X_test_le],axis=1)

#Ensure all columns have string types
X_train_oh.columns = X_train_oh.columns.astype(str)
X_test_oh.columns = X_test_oh.columns.astype(str)

X_train_oh.head()

In [50]:
#Checking above approach's MAE

print('MAE after converting all the categorical data into numerical data:')
print(score_dataset(X_train_oh,X_test_oh,y_train,y_test))

MAE after converting all the categorical data into numerical data:
MAE from XGB = 9875.460877403846
MAE from LinearRegressor = 14472.635991617191
None
